# PersistBench — Gate G2 propagation smoke (Kaggle)

Runs `scripts/g2_smoke.py` with `configs/g2_smoke.json`: Qwen2.5-7B-Instruct + Qwen3-8B (the two G1-cleared arms) x 2 chains (chain01 tracking, chain05 stock) x 2 faults (silent_wrong_value, stale_value) x 3 policies (M0/M1/M5), dependent mode revising, seed 0 = **24 chain runs, 72 dependent outcomes**.

**Reuse, don't regenerate.** The 24 origin trajectories + stored writes already exist in the pilot outputs (`outputs/pilot_writes.jsonl` for Qwen2.5; `results/qwen3-repilot/pilot_writes.jsonl` for Qwen3 — both tracked, so the clone below has them). The smoke generates **only the 72 dependent answers live** — minutes on a T4, not a second pilot. If a store cannot be reused, the script writes `fallback_pilot_config.json` and exits non-zero; the fallback (full regeneration of the 24 origins + writes) is documented in `persistbench/g2_smoke.py` and is NOT run here.

Gate G2 (design note `docs/g2-propagation-smoke-design.md`) is a **measurability smoke, not an effect gate**: PASS needs answer yield >= 80%, the M0 floor and M5 ceiling to hold, no cell errors with the correction present in every Session-4 prompt, and a computable propagation contrast on the G6-eligible subset. A clean null (M1 downstream ~= M0 downstream, floor and ceiling holding) is a reportable outcome, not a FAIL.

**Before launching:** commit + push this prep (the notebook clones the repo at its default branch — it must contain `scripts/g2_smoke.py`, `configs/g2_smoke.json`, and `persistbench/g2_smoke.py`), and read the current GPU quota first. The smoke queues after any running Kaggle work; it is short, so it can follow a queued run the same day.


## 1. GPU check — hard fail without CUDA

The smoke models are 7-8B at 4-bit; a CPU run would take hours and a silent Accelerator=None commit must never burn a session. This cell raises if no GPU is visible.


In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NO GPU: set Accelerator = GPU T4 x2 before Save & Run All, then re-run.'
print('device:', torch.cuda.get_device_name(0))

## 2. Get the code

Option A: public GitHub repo (set the URL). Option B: the repo uploaded into the working directory. The cell clones into `/kaggle/working/persistbench` or uses the uploaded copy, then verifies the G2 prep is present (the smoke script, its config, and the two pilot writes files the smoke reuses).


In [ ]:
import os, subprocess

REPO_URL = os.environ.get('PERSISTBENCH_REPO_URL', 'https://github.com/karmendra8386/persistbench-bench.git')  # published Nov 1, 2026; env var overrides
DEST = '/kaggle/working/persistbench'
if os.path.isdir(os.path.join(DEST, 'persistbench')):
    print('using uploaded repo copy at', DEST)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, DEST], check=True)
    print('cloned', REPO_URL, '->', DEST)

need = ['scripts/g2_smoke.py', 'configs/g2_smoke.json', 'persistbench/g2_smoke.py',
        'outputs/pilot_writes.jsonl', 'results/qwen3-repilot/pilot_writes.jsonl']
missing = [f for f in need if not os.path.exists(os.path.join(DEST, f))]
assert not missing, ('G2 prep missing from the repo copy — commit + push it first, or upload it: ' + ', '.join(missing))
print('G2 prep present; pilot writes files found:')
for f in need[3:]:
    print(' ', f, os.path.getsize(os.path.join(DEST, f)), 'bytes')
print(subprocess.run(['git', '-C', DEST, 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip(), '(repo HEAD)')

## 3. Install dependencies — preserving Kaggle's CUDA torch

Kaggle's image ships the CUDA torch build the smoke needs; a naive install can replace it. The packages below do not pin torch, and the cell re-checks CUDA afterwards. (transformers 5.x requires accelerate for device_map; bitsandbytes supplies the 4-bit quantisation.)


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'bitsandbytes'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA lost after install — stop here, do not run the smoke.'
print('deps installed; cuda still available')

## 4. Run the smoke

24 chain runs (2 models x 2 chains x 2 faults x 3 policies), 72 dependent answers, seed 0, revising dependents. Expected wall time: minutes on a T4 x2 — the origin trajectories and stored writes are reused from the pilot outputs, so no formation GPU is spent. Results land in `/kaggle/working/g2-smoke`.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/g2_smoke.py',
     '--config', 'configs/g2_smoke.json',
     '--out', '/kaggle/working/g2-smoke'],
    cwd='/kaggle/working/persistbench',
)
print('smoke exit code:', result.returncode)
print('exit 0 = G2 PASS; exit 1 = G2 FAIL (see summary); exit 2 = reuse failure, fallback config written — do NOT re-run blindly, read the message first')

## 5. Outputs — retrieval list

Expected files in `/kaggle/working/g2-smoke` (also the version Output tab): `g2_smoke_summary.md` (gate verdict + reported rates), `g2_smoke_summary.json`, `g2_smoke_cells.csv`, `g2_smoke_cells.jsonl`, and `transcripts/` (one JSON per dependent outcome: prompt, transcript, parse log, usage). On a reuse failure the directory instead holds `fallback_pilot_config.json` — retrieve it, do not re-run.


In [ ]:
import os

OUT = '/kaggle/working/g2-smoke'
for name in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, name)
    size = os.path.getsize(path) if os.path.isfile(path) else len(os.listdir(path))
    print(f'{name:32} {size:>10} {"bytes" if os.path.isfile(path) else "files"}')

print()
print(open(os.path.join(OUT, 'g2_smoke_summary.md')).read())

## After retrieval (not in this notebook)

1. Copy `/kaggle/working/g2-smoke` into the repo as `results/g2-smoke/` and commit it.
2. Write the G2 verdict record (`docs/g2-smoke-results.md`) from the summary: gate verdict, answer yield, M0/M5 floor-ceiling, propagation / persistence / correction-resistance on the eligible subset with Wilson CIs, per model only — plus an amendment §I entry recording the verdict and the H4 narrowing (live dependent does not condition on provenance; M2 not in the smoke set).
3. File the verdict record to Drive §03 + update the EB1 tracker in the same pass (repo rule).
4. Only on G2 PASS: proceed toward the two-model scale-up under gates G3–G7. No scale-up spend beyond the smoke until G2 passes.
